# Function Calling / Tool Use with Llama 3.1 (Groq)

**Author:** Ibrahim  
**Environment:** Google Colab / Python 3

## Overview
This notebook demonstrates **function calling (tool use)** – a state‑of‑the‑art capability that allows an LLM to decide when to call external functions, extract parameters, and use the results to answer user queries. We use Groq’s Llama 3.3 70B model, which has native support for tool calling.

## What You Will Build
- A set of tools:
  - `get_current_weather` (mock, returns sample weather)
  - `calculate` (evaluates simple math expressions)
- The LLM will autonomously decide which tool to call, extract arguments, and incorporate the results into its final answer.

## Why This Matters
Function calling is the foundation of agentic AI systems – chatbots that can look up information, perform calculations, interact with APIs, and execute actions based on natural language instructions.

## Requirements
- **Groq API key** (free tier from [console.groq.com](https://console.groq.com))

---

**© 2026 Ibrahim – Agentic function calling with Groq.**

### Install Dependencies

In [1]:
!pip install -q groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 15.5 MB/s eta 0:00:00


### Imports & API Key Setup

In [2]:
import json
from getpass import getpass
from groq import Groq

# Securely input your Groq API key
GROQ_API_KEY = getpass("Enter your Groq API key: ")
client = Groq(api_key=GROQ_API_KEY)

print("Groq client ready.")

Enter your Groq API key: ··········
Groq client ready.


### Define Tools (Function Specifications)

In [3]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": "Get the current weather in a given location",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city and state, e.g., San Francisco, CA"
                    },
                    "unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "The temperature unit to use"
                    }
                },
                "required": ["location"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Evaluate a mathematical expression",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "The mathematical expression to evaluate, e.g., '2 + 2 * 3'"
                    }
                },
                "required": ["expression"]
            }
        }
    }
]

print("Tools defined.")

Tools defined.


### Tool Implementations

In [4]:
def get_current_weather(location, unit="celsius"):
    # Mock implementation – in production, call a real weather API
    return {
        "location": location,
        "temperature": 22,
        "unit": unit,
        "condition": "sunny"
    }

def calculate(expression):
    try:
        # Use eval safely – only for demonstration, restrict in production
        result = eval(expression, {"__builtins__": {}}, {})
        return {"expression": expression, "result": result}
    except Exception as e:
        return {"error": str(e)}

available_functions = {
    "get_current_weather": get_current_weather,
    "calculate": calculate,
}

print("Tool implementations ready.")

Tool implementations ready.


### Function Calling Loop

In [5]:
def ask_with_tools(user_message):
    messages = [
        {
            "role": "system",
            "content": "You are a helpful assistant that can use tools to answer questions. When you need information from a tool, call it and use the result in your final answer."
        },
        {"role": "user", "content": user_message}
    ]

    # First call: model may decide to call a function
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=messages,
        tools=tools,
        tool_choice="auto",
        temperature=0
    )

    response_message = response.choices[0].message
    tool_calls = response_message.tool_calls

    if tool_calls:
        # Process each tool call
        messages.append(response_message)
        for tool_call in tool_calls:
            function_name = tool_call.function.name
            function_args = json.loads(tool_call.function.arguments)
            print(f" Calling tool: {function_name} with args {function_args}")

            if function_name in available_functions:
                function_response = available_functions[function_name](**function_args)
            else:
                function_response = {"error": f"Unknown tool: {function_name}"}

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(function_response)
            })

        # Second call: model uses the tool outputs to generate final answer
        second_response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=messages,
            temperature=0
        )
        final_answer = second_response.choices[0].message.content
    else:
        final_answer = response_message.content

    return final_answer

### Interactive Test

In [6]:
print("LLM Function Calling Demo")
print("Ask questions that need weather, calculations, or general info.")
print("Type 'exit' to quit.")

while True:
    user_input = input("\n Your question: ").strip()
    if user_input.lower() == "exit":
        break
    if not user_input:
        continue
    answer = ask_with_tools(user_input)
    print(f"\n {answer}\n")

LLM Function Calling Demo
Ask questions that need weather, calculations, or general info.
Type 'exit' to quit.

 Your question: Tell me the weather
 Calling tool: get_current_weather with args {'location': 'San Francisco, CA', 'unit': 'fahrenheit'}

 The current weather in San Francisco, CA is sunny with a temperature of 22 degrees Fahrenheit.


 Your question: exit


### Example Demonstrations

In [7]:
examples = [
    "What is the weather in London?",
    "Calculate 25 * 14 + 8",
    "What is the weather in Tokyo in fahrenheit?"
]

print("Running example queries:\n")
for ex in examples:
    print(f" {ex}")
    result = ask_with_tools(ex)
    print(f" {result}\n")

Running example queries:

 What is the weather in London?
 Calling tool: get_current_weather with args {'location': 'London', 'unit': 'celsius'}
 The current weather in London is sunny with a temperature of 22 degrees Celsius.

 Calculate 25 * 14 + 8
 Calling tool: calculate with args {'expression': '25 * 14 + 8'}
 The result of the calculation 25 * 14 + 8 is 358.

 What is the weather in Tokyo in fahrenheit?
 Calling tool: get_current_weather with args {'location': 'Tokyo', 'unit': 'fahrenheit'}
 The current weather in Tokyo is sunny with a temperature of 22 degrees Fahrenheit.



### Final Summary

In [8]:
print("Function Calling / Tool Use - COMPLETED")
print("Author: Ibrahim")
print(" The LLM autonomously decides when to call tools.")
print(" Extracts arguments from natural language.")
print(" Uses mock weather and calculation tools.")
print(" Production‑ready pattern for agentic AI.")

Function Calling / Tool Use - COMPLETED
Author: Ibrahim
 The LLM autonomously decides when to call tools.
 Extracts arguments from natural language.
 Uses mock weather and calculation tools.
 Production‑ready pattern for agentic AI.
